# Simulating models

## Preparation

In [1]:
from cobra.io import read_sbml_model

In [2]:
model = read_sbml_model('data/iJO1366.xml.gz')

In [4]:
model.solver = "glpk"

## Parsimonious flux balance analysis (pFBA) (5)

In [5]:
from cobra.flux_analysis import pfba

pFBA is the same as regular FBA except that our objective instead of maximizing the growth rate, is to minimize the total sum of absolute flux values. Since this on its own would lead to the trivial solution of all fluxes equal to zero (death), we also have the constraint that the original objective must be some fraction of its FBA solution.

In [6]:
fba_solution = model.optimize()
pfba_solution = pfba(model, fraction_of_optimum=0.95)

In [7]:
print(fba_solution.fluxes.BIOMASS_Ec_iJO1366_core_53p95M)
print(fba_solution.fluxes.abs().sum())

0.9823718127269633
709.9877713224889


In [8]:
print(pfba_solution.fluxes.BIOMASS_Ec_iJO1366_core_53p95M)
print(pfba_solution.fluxes.abs().sum())

0.9332532220906267
663.8480001666475


## Flux variability analysis (10 + 10)

### Preparation

In [9]:
from cobra.flux_analysis import flux_variability_analysis
import escher

In [10]:
core_model = read_sbml_model('data/e_coli_core.xml.gz')

In [12]:
core_model.solver = "glpk"

### Flux variability analysis

Calculate all possible maximum and minimum fluxes for all reactions using FVA

In [13]:
result = flux_variability_analysis(core_model)

Inspect the result.

In [14]:
result

,minimum,maximum
ACALD,-8.345524e-14,0.000000e+00
ACALDt,-7.902708e-14,0.000000e+00
ACKr,-1.169900e-13,0.000000e+00
ACONTa,6.007250e+00,6.007250e+00
ACONTb,6.007250e+00,6.007250e+00
...,...,...
TALA,1.496984e+00,1.496984e+00
THD2,0.000000e+00,4.942351e-12
TKT1,1.496984e+00,1.496984e+00
TKT2,1.181498e+00,1.181498e+00


Get an overview of a few key statistics of the resulting flux ranges.

In [15]:
result.describe()

,minimum,maximum
count,9.500000e+01,9.500000e+01
mean,2.627753e+00,2.357377e+01
std,1.069810e+01,1.439744e+02
min,-2.917583e+01,-2.917583e+01
25%,-2.922403e-14,0.000000e+00
50%,0.000000e+00,7.897284e-13
75%,4.959985e+00,5.012180e+00
max,4.551401e+01,1.000000e+03


Visualize the flux ranges on a pathway map of _E. coli's_ central carbon metabolism.

In [24]:
abs_flux_ranges = abs(result.maximum - result.minimum).to_dict()
escher.Builder('e_coli_core.Core metabolism', reaction_data=abs_flux_ranges).display_in_notebook()

Exception: display_in_notebook is deprecated. The Builder is now a Jupyter Widget, so you can return the Builder in a cell to see it, or use the IPython display function (see Escher docs for details)

Those reactions showing up in red are futile cyles.

In [17]:
result[result.maximum > 500]

,minimum,maximum
FRD7,0.000000,994.935624
SUCDi,5.064376,1000.000000


### Loopless FVA

We add constraints to get rid of the loops using [cycle free flux](https://academic.oup.com/bioinformatics/article-lookup/doi/10.1093/bioinformatics/btv096).

In [16]:
loopless_result = flux_variability_analysis(core_model, loopless=True)

In [25]:
abs_flux_ranges = abs(loopless_result.maximum - loopless_result.minimum).to_dict()
escher.Builder('e_coli_core.Core metabolism', reaction_data=abs_flux_ranges).display_in_notebook()

Exception: display_in_notebook is deprecated. The Builder is now a Jupyter Widget, so you can return the Builder in a cell to see it, or use the IPython display function (see Escher docs for details)

### Further constraints to the FVA

FVA examines how extreme the fluxes for each reaction can be. This includes solutions with very low growth rate, or with very high total flux, that means solutions that are quite unlikely for an actual organism. We can add additional constraints to the FVA to make it a bit more realistic using the parameters `fraction_of_optimium` which sets growth rate to not go below a given ratio of the max (e.g. 0.9 which implies growth rate should not be less than 90% of the previous optimum), or `pfba_factor` which says we are not interested in solutions that have very high total absolute flux (e.g. 1.1 which implies no more than 10% over the minimum total flux).

In [18]:
flux_variability_analysis(core_model, fraction_of_optimum=0.9, loopless=True)

,maximum,minimum
ACALD,0.0000,-2.5424
ACALDt,0.0000,-2.5424
ACKr,0.0000,-3.8136
ACONTa,8.7538,3.3709
ACONTb,8.7538,3.3709
...,...,...
TALA,3.1389,0.0000
THD2,0.8068,0.8068
TKT1,3.1389,0.0000
TKT2,2.8549,-0.3118


In [26]:
from cobra.flux_analysis import find_blocked_reactions 
flux_variability_analysis(core_model, fraction_of_optimum=0.7, pfba_factor=1.1, loopless=True)
blocked = find_blocked_reactions(core_model)
print(f'number of blocked reactions: {len(blocked)}')

number of blocked reactions: 8


### Exercises

Modify the code to explore flux ranges for $\mu \gt 0.7 \ h^{-1}$ 

Using FVA, determine all blocked reactions ($ v = 0 $) in the model.

## Production envelopes (5)

Production envelopes are useful to illustrate how max and min fluxes for a given reaction varies when you fix flux of another reaction to a given value. E.g, "how much can product can I get given different growth rates?"

In [18]:
from cobra.flux_analysis import production_envelope

In [27]:
prod_env = production_envelope(model, reactions=model.reactions.BIOMASS_Ec_iJO1366_core_53p95M, 
                               objective=model.reactions.EX_ac_e)

In [28]:
prod_env

,carbon_source,flux_minimum,carbon_yield_minimum,mass_yield_minimum,flux_maximum,carbon_yield_maximum,mass_yield_maximum,BIOMASS_Ec_iJO1366_core_53p95M
0,EX_glc__D_e,0.0,0.0,0.0,29.093467,0.969782,0.953505,0.000000
1,EX_glc__D_e,0.0,0.0,0.0,27.702277,0.923409,0.907910,0.051704
2,EX_glc__D_e,0.0,0.0,0.0,26.311087,0.877036,0.862316,0.103408
3,EX_glc__D_e,0.0,0.0,0.0,24.919897,0.830663,0.816721,0.155111
4,EX_glc__D_e,0.0,0.0,0.0,23.528707,0.784290,0.771126,0.206815
5,EX_glc__D_e,0.0,0.0,0.0,22.137517,0.737917,0.725532,0.258519
6,EX_glc__D_e,0.0,0.0,0.0,20.746327,0.691544,0.679937,0.310223
7,EX_glc__D_e,0.0,0.0,0.0,19.352198,0.645073,0.634246,0.361926
8,EX_glc__D_e,0.0,0.0,0.0,17.955282,0.598509,0.588464,0.413630
9,EX_glc__D_e,0.0,0.0,0.0,16.553583,0.551786,0.542525,0.465334


In [30]:
%matplotlib inline

In [31]:
prod_env[prod_env.direction == 'maximum'].plot(kind='line', x='BIOMASS_Ec_iJO1366_core_53p95M', y='carbon_yield')

AttributeError: 'DataFrame' object has no attribute 'direction'

## Gene and reaction essentiality (5 + 5)

Which genes and reactions can knock-out and which are essential?

In [32]:
from cobra.flux_analysis import single_gene_deletion, single_reaction_deletion

In [ ]:
gene_deletions = single_gene_deletion(model)
print(f'Genes: {gene_deletions}')
reaction_deletions = single_reaction_deletion(model)
print(f'Reactions: {reaction_deletions}')

Genes:           ids    growth   status
0     {b3540}  0.982372  optimal
1     {b4152}  0.982372  optimal
2     {b0394}  0.982372  optimal
3     {b2378}  0.982372  optimal
4     {b0485}  0.982372  optimal
...       ...       ...      ...
1362  {b4227}  0.982372  optimal
1363  {b4130}  0.982372  optimal
1364  {b1714}  0.982372  optimal
1365  {b0033}  0.982372  optimal
1366  {b0243}  0.981472  optimal

[1367 rows x 3 columns]
reactions:                 ids    growth   status
0            {GAPD}  0.856060  optimal
1        {3HCINNMH}  0.982372  optimal
2     {EX_malthx_e}  0.982372  optimal
3       {ASCBptspp}  0.982372  optimal
4       {CDAPPA141}  0.982372  optimal
...             ...       ...      ...
2578     {EX_gmp_e}  0.982372  optimal
2579       {CO2tpp}  0.918972  optimal
2580    {MLDCP1Bpp}  0.982372  optimal
2581      {AGMt2pp}  0.982372  optimal
2582       {GHBDHx}  0.982372  optimal

[2583 rows x 3 columns]


A gene/reaction can be considered essential if removing it leads to an infeasible model, or a model with very low growth rate.

In [ ]:
set(gene_deletions[(gene_deletions.flux < 0.01) | (gene_deletions.status != 'optimal')].index)
set(reaction_deletion[(reaction_deletions.flux < 0.01) | (reaction_deletions.statue != 'optimal')].index) 

AttributeError: 'DataFrame' object has no attribute 'flux'

Or use the convenience function

In [39]:
from cobra.flux_analysis import find_essential_genes, find_essential_reactions
find_essential_genes(model)
print('reactions')
find_essential_reactions(model)

reactions


{<Reaction 3OAR140 at 0x201084d6c30>,
 <Reaction 3OAS140 at 0x201084fce30>,
 <Reaction 5DOAN at 0x20108525730>,
 <Reaction A5PISO at 0x20108525d90>,
 <Reaction ACCOAC at 0x20108572240>,
 <Reaction ACGK at 0x20108573350>,
 <Reaction ACGS at 0x20108573c80>,
 <Reaction ACHBS at 0x20108573e30>,
 <Reaction ACLS at 0x201085b8320>,
 <Reaction ACODA at 0x201085ba750>,
 <Reaction ACONTa at 0x201085bae70>,
 <Reaction ACONTb at 0x201085baa20>,
 <Reaction ACOTA at 0x201085bb380>,
 <Reaction ADCL at 0x201086051f0>,
 <Reaction ADCS at 0x20108605400>,
 <Reaction ADSK at 0x201086078f0>,
 <Reaction ADSL1r at 0x201086073b0>,
 <Reaction ADSL2r at 0x201086079e0>,
 <Reaction ADSS at 0x20108607c80>,
 <Reaction AGPAT160 at 0x20108649e80>,
 <Reaction AGPAT161 at 0x20108649fd0>,
 <Reaction AGPR at 0x2010864a3c0>,
 <Reaction AHCYSNS at 0x2010864a4b0>,
 <Reaction AICART at 0x2010864aa80>,
 <Reaction AIRC2 at 0x2010864ac00>,
 <Reaction AIRC3 at 0x2010864ad50>,
 <Reaction ALAALAr at 0x2010864b3e0>,
 <Reaction ALAR

### Exercise

1. Find the essential reactions instead of genes.
2. MOMA is a common approach simulation approach to use instead of FBA when doing knock-out simulations. Briefly, MOMA adds constraints to minimize the deviation from the state without the knock-out, causing smaller effects. Use `method='linear moma'` to simulate MOMA gene deletions.